In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install cuml-cu12 --extra-index-url=https://pypi.nvidia.com

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    matthews_corrcoef,
)
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedGroupKFold,
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline
import joblib

In [ ]:
DIR_PATH = '/content/drive/MyDrive/1. Academics/ENEE408N/ENEE408N Project/sample of DREAMT dataset'
# DIR_PATH = '/data'
TRAIN_PATH = os.path.join(DIR_PATH, 'unbal_all_ppg_feat_extr_train.csv')
TEST_PATH = os.path.join(DIR_PATH, 'unbal_all_ppg_feat_extr_test.csv')

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

X_train = train_df.drop(['Label'], axis=1)
y_train = train_df['Label']
X_test = test_df.drop(['Label'], axis=1)
y_test = test_df['Label']

A = sum(y_train)
X = len(y_train)
print(f'Apnea count: {A}/{X} ({100*A/X:.2f}%)')
print(f'Nonapnea count: {X-A}/{X} ({100*(X-A)/X:.2f}%)')

ros = RandomOverSampler(random_state=42)
X_train, y_train = ros.fit_resample(X_train, y_train)

A = sum(y_train)
X = len(y_train)
print(f'Apnea count: {A}/{X} ({100*A/X:.2f}%)')
print(f'Nonapnea count: {X-A}/{X} ({100*(X-A)/X:.2f}%)')

# ML Models

In [ ]:
def model_pipeline(X_train, y_train, model, scaler=None, dim_reduction=None, name=None):
  if scaler is None and dim_reduction is None:
    pipeline = ImbPipeline([
        ("model", model)
    ])
  elif scaler is None:
    pipeline = ImbPipeline([
        ("dim_reduction", dim_reduction),
        ("model", model)
    ])
  elif dim_reduction is None:
    pipeline = ImbPipeline([
        ("scaler", scaler),
        ("model", model)
    ])
  else:
    pipeline = ImbPipeline([
        ("scaler", scaler),
        ("dim_reduction", dim_reduction),
        ("model", model)
    ])

  pipeline.fit(X_train, y_train)

  if name is not None:
    joblib.dump(pipeline, os.path.join(DIR_PATH, f'{name}.joblib'))

  return pipeline

def model_report(pipeline, X_test, y_test, show=False):
  y_pred = pipeline.predict(X_test)

  metrics = {}
  metrics['accuracy'] = accuracy_score(y_test, y_pred)
  metrics['conf_mat'] = confusion_matrix(y_test, y_pred)
  metrics['classification_rep'] = classification_report(y_test, y_pred)

  if show:
    print("Accuracy:", metrics['accuracy'])
    print("Confusion matrix:\n", metrics['conf_mat'])
    print("Classification report:\n", metrics['classification_rep'])

  return metrics

## KNN

In [ ]:
pipeline = model_pipeline(
    X_train,
    y_train,
    model=KNeighborsClassifier(n_neighbors=3),
    scaler=StandardScaler(),
    name='knn_3'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

KNN + Univariate

In [ ]:
pipeline = model_pipeline(
    X_train,
    y_train,
    model=KNeighborsClassifier(n_neighbors=3),
    scaler=StandardScaler(),
    dim_reduction=SelectKBest(k=5),
    name='knn_3_univariate_5'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

KNN + Chi_square

In [ ]:
pipeline = model_pipeline(
    X_train,
    y_train,
    model=KNeighborsClassifier(n_neighbors=3),
    scaler=MinMaxScaler(),
    dim_reduction=SelectKBest(score_func=chi2, k=5),
    name='knn_3_chi_5'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

KNN + PCA

In [ ]:
pipeline = model_pipeline(
    X_train,
    y_train,
    model=KNeighborsClassifier(n_neighbors=3),
    scaler=StandardScaler(),
    dim_reduction=PCA(n_components=3),
    name='knn_3_pca_3'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

KNN + LDA

In [ ]:
pipeline = model_pipeline(
    X_train,
    y_train,
    model=KNeighborsClassifier(n_neighbors=3),
    scaler=StandardScaler(),
    dim_reduction=LinearDiscriminantAnalysis(n_components=1),
    name='knn_3_lda_1'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

## RF

RF + Univariate

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=None,
    min_samples_leaf=5
)
pipeline = model_pipeline(
    X_train,
    y_train,
    model=rf,
    scaler=StandardScaler(),
    dim_reduction=SelectKBest(k=5),
    name='rf_100w_univariate_5'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

RF + chi

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=None,
    min_samples_leaf=5
)
pipeline = model_pipeline(
    X_train,
    y_train,
    model=rf,
    scaler=MinMaxScaler(),
    dim_reduction=SelectKBest(score_func=chi2, k=5),
    name='rf_100w_chi_5'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

RF + PCA

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=None,
    min_samples_leaf=5
)
pipeline = model_pipeline(
    X_train,
    y_train,
    model=rf,
    scaler=StandardScaler(),
    dim_reduction=PCA(n_components=3),
    name='rf_100w_pca_3'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

RF + LDA

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=None,
    min_samples_leaf=5
)
pipeline = model_pipeline(
    X_train,
    y_train,
    model=rf,
    scaler=StandardScaler(),
    dim_reduction=LinearDiscriminantAnalysis(n_components=1),
    name='rf_100w_lda_1'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

In [ ]:
import os
# Check it exists and see its size
path = os.path.join(DIR_PATH, 'rf_100w_lda_1.joblib')
print(os.path.exists(path))        # should be True
print(os.path.getsize(path), "bytes")

# Download
from google.colab import files
files.download(path)

## SVM

SVM + univariate

In [ ]:
pipeline = model_pipeline(
    X_train,
    y_train,
    model=SVC(kernel='rbf'),
    scaler=StandardScaler(),
    dim_reduction=SelectKBest(k=5),
    name='svm_rbf_univariate_5'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

SVM + chi2

In [ ]:
pipeline = model_pipeline(
    X_train,
    y_train,
    model=SVC(kernel='rbf'),
    scaler=MinMaxScaler(),
    dim_reduction=SelectKBest(score_func=chi2, k=5),
    name='svm_rbf_chi_5'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

SVM + PCA

In [ ]:
pipeline = model_pipeline(
    X_train,
    y_train,
    model=SVC(kernel='rbf'),
    scaler=StandardScaler(),
    dim_reduction=PCA(n_components=3),
    name='svm_rbf_pca_3'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

SVM + LDA

In [ ]:
pipeline = model_pipeline(
    X_train,
    y_train,
    model=SVC(kernel='rbf'),
    scaler=StandardScaler(),
    dim_reduction=LinearDiscriminantAnalysis(n_components=1),
    name='svm_rbf_lda_1'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

# Feedforward Neural Net